### DATA TRAINER MODULE

### Workflows


1. **Config.yaml**: Este archivo contendrá toda la configuración relacionada con el proyecto. Se utilizará para almacenar todos los parámetros relacionados con el proyecto, así como todas las rutas del proyecto. También se usará para guardar todos los hiperparámetros.
2. **Params.yaml**: Este archivo contendrá todos los hiperparámetros relacionados con el proyecto. Se utilizará para almacenar los hiperparámetros del proyecto y, específicamente, los hiperparámetros del entrenamiento del modelo.
3. **Entidad de configuración (Config entity)**: Este componente definirá la estructura de los archivos de configuración y proporcionará una forma de acceder a los parámetros de configuración de manera estructurada.
4. **Administrador de configuración (Configuration Manager)**: Este componente gestionará la carga y actualización de los archivos de configuración, asegurando que se utilice la configuración correcta en todo el proyecto.
5. **Actualizar componentes**: Ingesta de datos, Transformación de datos, Entrenador del modelo.
6. **Crear pipelines**: Pipeline de entrenamiento, Pipeline de predicción.

In [1]:
import os
%pwd

'c:\\Users\\mecenturion\\Desktop\\MARTIN\\UDEMY\\MLOPS\\TextSummarizer\\textsummarizer\\research'

In [2]:
os.chdir("..")
%pwd

'c:\\Users\\mecenturion\\Desktop\\MARTIN\\UDEMY\\MLOPS\\TextSummarizer\\textsummarizer'

In [35]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    data_path: Path
    model_path: Path
    tokenizer_path: Path
    metric_file_name: Path

In [45]:
from src.textSummarizer.constants import *
from src.textSummarizer.utils.common import read_yaml, create_directories
from src.textSummarizer.logging import logger

# Creamos el configuration mangarer, que se encargará de cargar el archivo de configuración y devolver la configuración de cada componente.
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        # Creamos el directorio de Artifacts, que es donde se guardarán todos los artefactos del proyecto.
        create_directories([self.config.artifacts_root])

    def get_evaluaton_config(self) -> ModelEvaluationConfig:
        evaluation_config = self.config.model_evaluation

        create_directories([evaluation_config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=Path(evaluation_config.root_dir),
            data_path=Path(evaluation_config.data_path),
            model_path=Path(evaluation_config.model_path),
            tokenizer_path=Path(evaluation_config.tokenizer_path),
            metric_file_name=Path(evaluation_config.metric_file_name)
        )

        return model_evaluation_config

In [46]:

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_from_disk
import torch
import pandas as pd
from tqdm.auto import tqdm

import evaluate


class ModelEvaluation:
    def __init__(self, config = ModelEvaluationConfig):
        self.config = config
    
    def generate_batch_sized_chunks(self, list_of_elements, batch_size):
        """Divide el dataset en fragmentos más pequeños para procesarlos en batches."""
        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i : i + batch_size]

    def calculate_metric_on_test_ds(self, dataset, metric, model, tokenizer,
                                    batch_size=16, device="cuda",
                                    column_text="dialogue",
                                    column_summary="summary"):

        # Dividir en batches
        article_batches = list(self.generate_batch_sized_chunks(dataset[column_text], batch_size))
        target_batches = list(self.generate_batch_sized_chunks(dataset[column_summary], batch_size))

        model.to(device)
        model.eval()

        for article_batch, target_batch in tqdm(
            zip(article_batches, target_batches), total=len(article_batches)
        ):

            # Tokenizar input
            inputs = tokenizer(
                article_batch,
                max_length=1024,
                truncation=True,
                padding="max_length",
                return_tensors="pt"
            )

            # ✅ CORRECCIÓN AQUÍ (usar inputs reales)
            summaries = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                length_penalty=0.8,
                num_beams=8,
                max_length=128
            )

            # Decodificar resúmenes generados
            decoded_summaries = [
                tokenizer.decode(s, skip_special_tokens=True, clean_up_tokenization_spaces=True)
                for s in summaries
            ]

            # Limpiar espacios extra
            decoded_summaries = [d.replace("", " ") for d in decoded_summaries]

            # Añadir resultados a la métrica
            metric.add_batch(predictions=decoded_summaries, references=target_batch)

            score = metric.compute()
        
        return score
    
    def evaluate(self):
        # Cargar dataset
        dataset_samsum_pt = load_from_disk(self.config.data_path)
        
        device = "cuda" if torch.cuda.is_available() else "cpu"
        # Cargar modelo y tokenizer
        model_path = Path(self.config.model_path)
        tokenizer_path = Path(self.config.tokenizer_path)

        model = AutoModelForSeq2SeqLM.from_pretrained(str(model_path)).to(device)
        tokenizer = AutoTokenizer.from_pretrained(str(tokenizer_path))

        # Cargar métrica ROUGE
        rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
        
        rouge_metric = evaluate.load("rouge")

        # Calcular métrica en el dataset de test
        scores = self.calculate_metric_on_test_ds(
            dataset=dataset_samsum_pt['test'][0:10],
            metric=rouge_metric,
            model=model,
            tokenizer=tokenizer,
            batch_size=2,
            device=device,
            column_text="dialogue",
            column_summary="summary"
        )

        rouge_dict = {rn: scores[rn] for rn in rouge_names}

        # Guardar resultados en un archivo CSV
        df_scores = pd.DataFrame(rouge_dict, index=['pegasus'])
        df_scores.to_csv(self.config.metric_file_name, index=False)

In [47]:
config = ConfigurationManager()
model_evaluation_config = config.get_evaluaton_config()
model_evaluation = ModelEvaluation(config=model_evaluation_config)
model_evaluation.evaluate()

[2026-04-22 11:50:37,122: INFO: common: 31: YAML file 'C:\Users\mecenturion\Desktop\MARTIN\UDEMY\MLOPS\TextSummarizer\textsummarizer\config\config.yaml' read successfully.]
[2026-04-22 11:50:37,127: INFO: common: 31: YAML file 'C:\Users\mecenturion\Desktop\MARTIN\UDEMY\MLOPS\TextSummarizer\textsummarizer\params.yaml' read successfully.]
[2026-04-22 11:50:37,131: INFO: common: 53: Directory 'artifacts' created successfully.]
[2026-04-22 11:50:37,134: INFO: common: 53: Directory 'artifacts/model_evaluation' created successfully.]


  0%|          | 0/5 [00:00<?, ?it/s]

[2026-04-22 11:52:27,109: INFO: rouge_scorer: 83: Using default tokenizer.]


 20%|██        | 1/5 [01:40<06:42, 100.61s/it]

[2026-04-22 11:56:17,035: INFO: rouge_scorer: 83: Using default tokenizer.]


...

 60%|██████    | 3/5 [08:23<05:49, 174.99s/it]

[2026-04-22 12:00:40,893: INFO: rouge_scorer: 83: Using default tokenizer.]


 80%|████████  | 4/5 [09:54<02:21, 141.70s/it]

[2026-04-22 12:01:46,746: INFO: rouge_scorer: 83: Using default tokenizer.]


100%|██████████| 5/5 [11:00<00:00, 132.03s/it]
